# Listenbrainz data preprocessing

## Import dependencies

In [2]:
import pandas as pd
from pathlib import Path
import uuid

## Load datasets in a dataframe

In [ ]:
df01 = pd.read_parquet('original/01.parquet')
df02 = pd.read_parquet('original/02.parquet')
df03 = pd.read_parquet('original/03.parquet')
df = pd.concat([df01, df02, df03], ignore_index=True)

## Preprocess data


The section below removes rows where either of the following are empty or not a valid value for the column:
listened_at, created, artist_name, release_name, release_mbid, recording_name, recording_mbid

In [4]:
required_cols = [
    'listened_at',
    'created',
    'artist_name',
    'release_name',
    'release_mbid',
    'recording_name',
    'recording_mbid',
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f'Missing required columns: {missing_cols}')

def valid_nonempty_text(series):
    s = series.astype('string')
    return s.notna() & s.str.strip().ne('')

def valid_uuid_value(v):
    if pd.isna(v):
        return False
    t = str(v).strip()
    if not t:
        return False
    try:
        uuid.UUID(t)
        return True
    except ValueError:
        return False

def valid_datetime_or_unix(series):
    parsed_direct = pd.to_datetime(series, errors='coerce', utc=True)
    parsed_unix = pd.to_datetime(pd.to_numeric(series, errors='coerce'), unit='s', errors='coerce', utc=True)
    return parsed_direct.notna() | parsed_unix.notna()

def getCleaningSummary(before_rows, after_rows):
    return {
        'before_rows': int(before_rows),
        'after_rows': int(after_rows),
        'removed_rows': int(before_rows - after_rows),
    }

valid_mask = (
    valid_datetime_or_unix(df['listened_at'])
    & valid_datetime_or_unix(df['created'])
    & valid_nonempty_text(df['artist_name'])
    & valid_nonempty_text(df['release_name'])
    & df['release_mbid'].apply(valid_uuid_value)
    & valid_nonempty_text(df['recording_name'])
    & df['recording_mbid'].apply(valid_uuid_value)
)

before_rows = len(df)
df = df.loc[valid_mask].copy()
after_rows = len(df)
getCleaningSummary(before_rows, after_rows)


{'before_rows': 3555850, 'after_rows': 3120417, 'removed_rows': 435433}

The following section removes consecutive listens of a same song.

In [6]:
required_cols = ['user_id', 'listened_at', 'recording_mbid']
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f'Missing required columns: {missing_cols}')

before_rows = len(df)

listened_at_ts = pd.to_datetime(df['listened_at'], errors='coerce', utc=True)
listened_at_unix = pd.to_datetime(
    pd.to_numeric(df['listened_at'], errors='coerce'),
    unit='s',
    errors='coerce',
    utc=True,
)
listened_at_sort = listened_at_ts.fillna(listened_at_unix)

df_sorted = (
    df.assign(_listened_at_sort=listened_at_sort)
    .sort_values(['user_id', '_listened_at_sort'], kind='mergesort')
)

prev_recording = df_sorted.groupby('user_id')['recording_mbid'].shift()
consecutive_same_song = df_sorted['recording_mbid'].eq(prev_recording)

df = df_sorted.loc[~consecutive_same_song].drop(columns=['_listened_at_sort']).reset_index(drop=True)

after_rows = len(df)
getCleaningSummary(before_rows, after_rows)


{'before_rows': 3120417, 'after_rows': 2987500, 'removed_rows': 132917}

The following section removes rows with recording_mbids that appear less than 5 times in the whole dataset.

In [7]:
before_rows = len(df)

recording_mbid_counts = df['recording_mbid'].value_counts()
valid_recording_mbids = recording_mbid_counts[recording_mbid_counts >= 5].index

df = df[df['recording_mbid'].isin(valid_recording_mbids)].copy()

after_rows = len(df)
getCleaningSummary(before_rows, after_rows)


{'before_rows': 2987500, 'after_rows': 2272160, 'removed_rows': 715340}

The following section removes sessions that contain only one song

In [ ]:
before_rows = len(df)

session_sizes = df.groupby(['user_id', 'created'])['recording_mbid'].transform('size')
df = df[session_sizes > 1].copy().reset_index(drop=True)

after_rows = len(df)
getCleaningSummary(before_rows, after_rows)


{'before_rows': 2272160, 'after_rows': 2186027, 'removed_rows': 86133}

## Preprocessed data summary

In [9]:
# Count unique MBIDs
unique_recording_mbids = df['recording_mbid'].nunique()
print(f'Unique recording MBIDs: {unique_recording_mbids}')

# Count unique release MBIDs
unique_release_mbids = df['release_mbid'].nunique()
print(f'Unique release MBIDs: {unique_release_mbids}')

# Count unique sessions
unique_sessions = df.groupby(['user_id', 'created']).ngroups
print(f'Unique sessions: {unique_sessions}')

Unique recording MBIDs: 118950
Unique release MBIDs: 40678
Unique sessions: 51566


## Output preprocessed data and save it to output

In [ ]:
MAX_FILE_MB = 100 # maximum size for direct upload to github
output_files = ['clean/01_clean.parquet', 'clean/02_clean.parquet', 'clean/03_clean.parquet']



# Split rows as evenly as possible across 3 parts.
n_parts = len(output_files)
n_rows = len(df)
base_size = n_rows // n_parts
remainder = n_rows % n_parts

parts = []
start = 0
for i in range(n_parts):
    end = start + base_size + (1 if i < remainder else 0)
    parts.append(df.iloc[start:end])
    start = end

# Format to zstd with level 12, then gzip with level 9 if still too large. Stop once we have a small enough file.
compression_attempts = [('zstd', 12), ('gzip', 9)]
size_report = []

for compression, level in compression_attempts:
    for part, output_file in zip(parts, output_files):
        part.to_parquet(
            output_file,
            index=False,
            engine='pyarrow',
            compression=compression,
            compression_level=level,
        )

    size_report = [
        (f, len(p), round(Path(f).stat().st_size / (1024 ** 2), 2))
        for f, p in zip(output_files, parts)
    ]

    if max(size_mb for _, _, size_mb in size_report) < MAX_FILE_MB:
        break

